### 1. Construct initial test text

In [1]:
filename = 'textV1.txt'

with open(filename, 'w') as file_object:
    file_object.write("low low low low low\n")
    file_object.write("lower lower widest widest widest\n")
    file_object.write("newest newest newest newest newest newest\n")

### 2. Initialization: GPT2-style character mapping function

In [31]:
def gpt2_bytes_to_unicode_local(): # Use local name to avoid potential conflicts
    """
    Convert bytes to Unicode characters. 
    Calling the function directly returns a dictionary in the format {number: unicode character}.
    """
    # bs: byte values (0-255)
    bs = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )
    # cs: corresponding unicode code points
    cs = bs[:]

    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    
    return dict(zip(bs, cs))

In [32]:
def validate_special_tokens(
    vocab: dict[int, str],
    special_tokens: list[str],
) -> list[str]:
    """
    Verify the validity of special characters and incorporate them into the original mapping dictionary.

    Args:
        vocab (dict[int, str]): Original dictionary
        special_tokens (list[str]): A list of string special tokens to be added to the tokenizer vocabulary.

    Returns:
        list[str],
            vocab_special_validated: Validated initial dictionary with special characters.
    """

    seen = set()
    current_next_id: int = 256 # New token IDs start from 256.
    for token in special_tokens:
        # Check for empty strings
        if not token:
            raise ValueError("Special symbols cannot be empty strings.")
        
        # Check for duplicate special symbols.
        if token in seen:
            raise ValueError(f"Duplicate special symbols.: '{token}'")
        seen.add(token)
        
        # Check if it has already been mapped in the initial vocabulary.
        if len(token) == 1 and ord(token) < 324:
            raise ValueError(f"Duplicate special tokens in the initial vocabulary.: '{token}'")

        # No issue, add it to the initial vocabulary.
        vocab[current_next_id] = token
        current_next_id += 1

    return vocab

Check the generated mapping tuples

In [37]:
# Initialization mapping
special_tokens=["<|endoftext|>"]
byte_to_unicode = validate_special_tokens(vocab=gpt2_bytes_to_unicode_local(), special_tokens=special_tokens)
# byte_to_unicode

### 3. Text Preprocessing

In [39]:
import re

input_path = filename

try:
    with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read() # 读取整个文件内容
except FileNotFoundError:
    text = "" # 如果文件不存在，视为空文本处理

# 对语料库里的文段进行简单的预分词：注意按要求要保留空格分割文本，得到“单词”列表["hello"," world"]
# raw_words: List[str] = text.split()
raw_words: list[str] = re.findall(r'\s*\S+', text)
raw_words

['low',
 ' low',
 ' low',
 ' low',
 ' low',
 '\nlower',
 ' lower',
 ' widest',
 ' widest',
 ' widest',
 '\nnewest',
 ' newest',
 ' newest',
 ' newest',
 ' newest',
 ' newest']

In [ ]:
# 然后把“单词”转换为初始的unicode字符序列,[[u'h', u'e', u'l', u'l', u'o'], [u'w', u'o', u'r', u'l', u'd']]
unicode_sequences: list[list[str]] = []

for word_str in raw_words:
    word_as_raw_bytes: bytes = word_str.encode("utf-8") # 把每个单词都转为字节串形式
    if not word_as_raw_bytes: # 跳过空字符串（可能由多个连续空格产生）
        continue
    unicode_sequences.append([byte_to_unicode[byte_val] for byte_val in word_as_raw_bytes]) # 将单词映射为unicode字符，并添加到unicode_sequences中

merges: list[tuple[bytes, bytes]] = [] # 用于存储合并操作记录

unicode_sequences

[['l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ġ', 'l', 'o', 'w'],
 ['Ċ', 'l', 'o', 'w', 'e', 'r'],
 ['Ġ', 'l', 'o', 'w', 'e', 'r'],
 ['Ġ', 'w', 'i', 'd', 'e', 's', 't'],
 ['Ġ', 'w', 'i', 'd', 'e', 's', 't'],
 ['Ġ', 'w', 'i', 'd', 'e', 's', 't'],
 ['Ċ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't'],
 ['Ġ', 'n', 'e', 'w', 'e', 's', 't']]